# Transfection results

Reads `analysis/` written by `analyze.ipynb` and writes user-facing `results/<sample>/` packs (XLSX tables + single-panel PNGs). Re-run this notebook after a plot-script change without repeating timeseries/AUC/fit.

This notebook owns sample names and merges them into `workspace/assay.json` as `samples[]`. It does not invent a Config signal channel: that comes from analyze (or from `analysis/PosN/ch*.csv` if assay.json has no channels yet). If `analysis.channels.signal` is missing, setup writes it as a one-element list from that resolved channel. Per-sample signal maps are not set here. Run `analyze.ipynb` first.


## Config

In [ ]:
from pathlib import Path

# Folder with analysis/ (from analyze.ipynb); results/<sample>/ is written here. Not the ND2/CZI file.
WORKSPACE = Path(r"Z:\\projects\LNPbinder\Experiments\20260731\Auswertung")

# Minutes between acquired frames (must match analyze.ipynb). 10.0 → t = 0, 10, 20, …
# Used for plot time axes. Interval in assay.json is written by analyze.ipynb, not here.
INTERVAL_MINUTES = 10.0

# One entry per sample. name is the folder under results/ (use filesystem-safe names).
# positions: zero-based field indices, same as roi/Pos{n} and analyze.
#   list(range(0, 40)) is positions 0 through 39 (Python half-open range), not 0..40 inclusive.
SAMPLES = [
    {"name": "A431_aiLNP_incubated", "positions": list(range(0, 40))},
    {"name": "A549_aiLNP_incubated", "positions": list(range(40, 80))},
    {"name": "A549_aiLNP", "positions": list(range(80, 121))},
    {"name": "A431_aiLNP", "positions": list(range(121, 159))},
]


## Setup

In [ ]:
import json

from transfection.core import workspace_analysis_dir
from transfection.services import plot_auc, plot_fit, plot_timeseries


def _inclusive_position_spec(positions):
    ordered = sorted({int(position) for position in positions})
    if not ordered:
        raise ValueError("No positions to serialize")
    parts = []
    start = prev = ordered[0]
    for value in ordered[1:]:
        if value == prev + 1:
            prev = value
            continue
        parts.append(f"{start}:{prev}" if start != prev else str(start))
        start = prev = value
    parts.append(f"{start}:{prev}" if start != prev else str(start))
    return ",".join(parts)


def _resolve_signal_channel(workspace, payload):
    analysis = payload.get("analysis") if isinstance(payload.get("analysis"), dict) else {}
    channels = analysis.get("channels") if isinstance(analysis.get("channels"), dict) else {}
    signal = channels.get("signal")
    if isinstance(signal, list) and signal:
        return int(signal[0])
    analysis_dir = workspace_analysis_dir(workspace)
    if not analysis_dir.is_dir():
        raise FileNotFoundError("No analysis/ directory. Run analyze.ipynb first.")
    found = sorted({int(path.stem[2:]) for path in analysis_dir.glob("Pos*/ch*.csv") if path.stem.startswith("ch") and path.stem[2:].isdigit()})
    if not found:
        raise FileNotFoundError("No analysis.channels in assay.json and no analysis/Pos*/ch*.csv. Run analyze.ipynb first.")
    return found[0]

workspace = WORKSPACE.expanduser().resolve()
if not workspace.is_dir():
    raise FileNotFoundError(f"Workspace not found: {workspace}")
assay_path = workspace / "assay.json"
payload = {}
if assay_path.is_file():
    payload = json.loads(assay_path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"{assay_path} must contain a JSON object")
signal_channel = _resolve_signal_channel(workspace, payload)
samples_payload = []
for slide_channel, row in enumerate(SAMPLES):
    name = str(row.get("name", "")).strip()
    positions = row.get("positions")
    samples_payload.append(
        {
            "slideChannel": slide_channel,
            "name": name,
            "positions": _inclusive_position_spec(positions),
        }
    )
payload["samples"] = samples_payload
analysis = payload.get("analysis") if isinstance(payload.get("analysis"), dict) else {}
channels = analysis.get("channels") if isinstance(analysis.get("channels"), dict) else {}
if not (isinstance(channels.get("signal"), list) and channels.get("signal")):
    channels["signal"] = [int(signal_channel)]
    channels.setdefault("mask", 0)
    analysis["channels"] = channels
    payload["analysis"] = analysis
assay_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
print(f"Workspace: {workspace}")
print(f"Signal channel: {signal_channel}")
print(f"Merged assay.json samples: {assay_path}")


## Plot timeseries

In [ ]:
for path in plot_timeseries.run_plot_timeseries(
    metrics_dir=workspace,
    interval=INTERVAL_MINUTES,
):
    print(plot_timeseries.format_written_timeseries_plot_message(path))


## Plot AUC

In [ ]:
for message in plot_auc.format_written_auc_plot_messages(
    list(plot_auc.run_plot_auc(auc_csv=workspace))
):
    print(message)


## Plot fit

In [ ]:
for message in plot_fit.format_written_fit_plot_messages(
    plot_fit.run_plot_fit(
        workspace,
        output=None,
        interval=INTERVAL_MINUTES,
        columns=None,
    )
):
    print(message)

print("Results finished.")
